# 00 · Подготовка и базовая линия

Три вещи, которые нужно увидеть до любого обучения:

1. модель грузится и отвечает;
2. коллатор открывает градиенту то, что нужно, и скрывает остальное;
3. как модель ведёт себя **до** вмешательства — иначе нечего будет сравнивать.

Остальные ноутбуки самодостаточны и каждый грузит модель сам. Этот — чтобы убедиться, что всё на месте.

In [ ]:
from common import MODEL_ID, SYSTEM, DATA, demo_answers, show, policy_suite, tools_suite, fmt, read_raw

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from vlmkit import ChatCollator, describe, load_jsonl, memory_report, preview, evaluate as ev

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
print(memory_report())

## Данные

Четыре набора в `data/`. В каждом есть отрицательная группа — без неё модель выучит поведение целиком и будет применять его везде.

In [ ]:
from collections import Counter

for name, key in (("policy.jsonl", "group"), ("tools.jsonl", "group"), ("skills.jsonl", "skill")):
    raw = read_raw(name)
    print(f"{name:<14} {len(raw):>3}  {dict(Counter(d[key] for d in raw))}")
print(f"{'prefs.jsonl':<14} {len(read_raw('prefs.jsonl')):>3}  пары chosen/rejected")

print()
print(describe(load_jsonl(DATA / "policy.jsonl"), processor))

## Что попадает в градиент

Обучаемые токены в ⟦скобках⟧. На траектории `multi` должно быть видно: обе реплики ассистента открыты, вопрос пользователя и результаты инструментов закрыты, блок `<think>` закрыт.

In [ ]:
tools = load_jsonl(DATA / "tools.jsonl")
tools_raw = read_raw("tools.jsonl")
multi = next(s for s, r in zip(tools, tools_raw) if r["group"] == "multi")
print(preview(multi, processor, system=SYSTEM))

## Базовая линия

Модель до всякого вмешательства. Эти же пять запросов повторяются в каждом следующем ноутбуке — по ним видно, что изменилось.

Метрики парные: попадание на целевых группах и ложные срабатывания на контрольных. Порознь обе бессмысленны.

In [ ]:
show(demo_answers(model, processor), "БЕЗ СИСТЕМНОГО ПРОМПТА")
show(demo_answers(model, processor, system=SYSTEM), "С СИСТЕМНЫМ ПРОМПТОМ")

print()
print("policy, без промпта:", fmt(ev.run(model, processor, policy_suite(system=None))))
print("policy, с промптом: ", fmt(ev.run(model, processor, policy_suite())))
print("tools:              ", fmt(ev.run(model, processor, tools_suite())))

## Что дальше

Запишите цифры базовой линии — они понадобятся для сравнения.

Порядок ноутбуков — от дешёвого к дорогому:

| | метод | что меняется |
|---|---|---|
| `02-steering` | вектор управления | ничего в весах, минуты |
| `01-sft` | LoRA на policy | адаптер, часы |
| `03-dpo` | выравнивание на парах | адаптер, часы |
| `04-tools` | LoRA на траекториях | адаптер, часы |
| `05-compare` | несколько методов подряд | сводная таблица |

Если вектор управления не даёт эффекта ни при каком коэффициенте — дообучение на тех же примерах скорее всего тоже не поможет, и это стоит узнать за минуты, а не за часы.